# ST · Ravi / Heiland (human glioblastoma, Visium)

Same pipeline as `st_01_vicari`, applied to 6 patients. Focus: what is **different** from mouse brain.
Read-only unless `SAVE = True` in the last section.

| | Vicari | Ravi/Heiland |
|---|---|---|
| Tissue | healthy mouse brain | human glioblastoma (tumour) |
| Slices | 3, one animal | 6 patients |
| Mito genes | `mt-` | `MT-` |
| Expected low-RNA biology | white matter | white matter **+ necrosis** |
| Expert labels | RegionLoupe (A1/B1) | none in `outs/` (2 extra CSVs to inspect) |

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

from stsm_integration.io import load_config, validate_adata
from stsm_integration.preprocess import st as pst

sc.settings.set_figure_params(dpi=80, frameon=False)

## 0 · Config, paths, Space Ranger summaries

In [ ]:
cfg = load_config("../../configs/ravi_heiland.yaml")
RAW = Path(cfg["paths"]["raw_st"])

def outs_dir(st_name: str) -> Path:
    p = RAW / st_name / "outs"
    if not p.exists():
        raise FileNotFoundError(p)
    return p

for s in cfg["slices"]:
    print(s["id"], "→", outs_dir(s["st"]))

Space Ranger writes a run summary per sample. Comparing them **before** any QC shows which patient is technically weaker.

In [ ]:
cols = ["Number of Spots Under Tissue", "Mean Reads per Spot", "Median Genes per Spot",
        "Median UMI Counts per Spot", "Fraction Reads in Spots Under Tissue", "Sequencing Saturation"]
summ = pd.concat({s["id"]: pd.read_csv(outs_dir(s["st"]) / "metrics_summary.csv").iloc[0]
                  for s in cfg["slices"]}, axis=1).T
summ[[c for c in cols if c in summ.columns]]

## 0b · Extra files
`Tab_GeneSet_*.csv` and `Top_GeneExpression_*.csv` exist only for UKF248 and UKF275. Look before deciding whether they matter.

In [ ]:
for s in cfg["slices"]:
    for f in sorted((outs_dir(s["st"]) / "spatial").glob("*_T.csv")):
        df = pd.read_csv(f)
        print(f"--- {s['id']} · {f.name} · shape {df.shape}")
        display(df.head(3))

## 1 · Load one patient

In [ ]:
SLICE = cfg["slices"][0]
adata = pst.load_visium(str(outs_dir(SLICE["st"])), SLICE["id"])
lib = next(iter(adata.uns["spatial"]))
print(adata)
print("scalefactors:", adata.uns["spatial"][lib]["scalefactors"])

## 2 · Coordinates → µm (sanity check: ~100 µm)

In [ ]:
pst.coords_to_um(adata)
xy = adata.obsm["spatial"]
d, _ = NearestNeighbors(n_neighbors=2).fit(xy).kneighbors(xy)
print(f"µm/px {adata.uns['um_per_px']:.3f} | extent {np.ptp(xy, axis=0).round(0)} µm | median NN {np.median(d[:, 1]):.1f} µm")

## 3 · QC on one patient

**Tumour-specific trap:** necrotic areas have few living cells, so low counts *and* often high mito. They are biology (a hallmark of glioblastoma), not failed spots. The v3 rule (absolute floors, flags for the rest) was designed for this; check it holds.

In [ ]:
adata.var["mt"] = adata.var_names.str.lower().str.startswith(cfg["st"]["mito_prefix"].lower())
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], layer="counts", percent_top=None, inplace=True)
print("mito genes:", adata.var["mt"].sum())

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].hist(np.log1p(adata.obs["total_counts"]), 60); ax[0].axvline(np.log1p(cfg["st"]["min_counts"]), c="r"); ax[0].set_title("log1p total counts")
ax[1].hist(np.log1p(adata.obs["n_genes_by_counts"]), 60); ax[1].axvline(np.log1p(cfg["st"]["min_genes"]), c="r"); ax[1].set_title("log1p genes")
ax[2].hist(adata.obs["pct_counts_mt"], 60); ax[2].set_title("% mito")
plt.tight_layout()

tmp = adata.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
sc.pl.spatial(tmp, color=["total_counts", "pct_counts_mt"], ncols=2)

In [ ]:
adata_raw = adata.copy()
adata = pst.qc_filter(adata, mito_prefix=cfg["st"]["mito_prefix"],
                      min_counts=cfg["st"]["min_counts"], min_genes=cfg["st"]["min_genes"],
                      mito_mode=cfg["st"]["mito_mode"])
adata.uns["qc"]

In [ ]:
# where are removed / flagged spots?
st_ = pd.Series("clean", index=adata_raw.obs_names, dtype=object)
st_[adata.obs_names[adata.obs["low_quality"]]] = "low_quality (kept)"
st_[adata.obs_names[adata.obs["high_mito"]]] = "high_mito (kept)"
st_[~adata_raw.obs_names.isin(adata.obs_names)] = "removed"
tmp = adata_raw.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
tmp.obs["qc_status"] = pd.Categorical(st_, categories=["clean", "low_quality (kept)", "high_mito (kept)", "removed"])
print(tmp.obs["qc_status"].value_counts())
sc.pl.spatial(tmp, color="qc_status", palette=["lightgrey", "gold", "crimson", "royalblue"])

In [ ]:
gs = pd.read_csv(outs_dir("#UKF248_T_ST") / "spatial" / "Tab_GeneSet_248_T.csv", index_col="barcodes")
adata.obs["hypoxia"] = adata.obs_names.map(gs["Reactive-Hypoxia"])
print(adata.obs.groupby("low_quality")["hypoxia"].median())
print(adata.obs.groupby("high_mito")["hypoxia"].median())

**Read the map:** if `low_quality` and `high_mito` overlap in one compact region, it is likely necrosis (check the H&E: pale, acellular). Kept + flagged is the right outcome.

## 4–5 · Normalise and contract

In [ ]:
pst.lognormalize(adata)
adata.uns["modality"] = "st"
adata.uns["pixel_size_um"] = cfg["st"]["spot_diameter_um"]
validate_adata(adata, "st")
print("X min/max:", adata.X.min(), adata.X.max(), "| contract OK")

## 6 · Quick look (not a reference)

In [ ]:
tmp = adata.copy(); tmp.obsm["spatial"] = tmp.obsm["spatial_px"]
sc.pp.highly_variable_genes(tmp, n_top_genes=2000, flavor="seurat_v3", layer="counts")
sc.pp.pca(tmp, mask_var="highly_variable")
sc.pp.neighbors(tmp)
sc.tl.leiden(tmp, resolution=0.5, flavor="igraph", n_iterations=2)
sc.pl.spatial(tmp, color=["leiden", "low_quality", "high_mito"], ncols=3)

## 7 · All patients: QC overview
One row per patient: histograms with floors + QC status map. Removed spots should look technical (edges, tears), never anatomical.

In [ ]:
rows, maps = [], []
for s in cfg["slices"]:
    raw = pst.load_visium(str(outs_dir(s["st"])), s["id"])
    pst.coords_to_um(raw)
    raw.var["mt"] = raw.var_names.str.lower().str.startswith(cfg["st"]["mito_prefix"].lower())
    sc.pp.calculate_qc_metrics(raw, qc_vars=["mt"], layer="counts", percent_top=None, inplace=True)
    kept = pst.qc_filter(raw.copy(), mito_prefix=cfg["st"]["mito_prefix"],
                         min_counts=cfg["st"]["min_counts"], min_genes=cfg["st"]["min_genes"],
                         mito_mode=cfg["st"]["mito_mode"])
    d, _ = NearestNeighbors(n_neighbors=2).fit(raw.obsm["spatial"]).kneighbors(raw.obsm["spatial"])
    q = kept.uns["qc"]
    rows.append({"patient": s["id"], "spots": q["spots_in"], "removed": q["spots_in"] - q["spots_out"],
                 "removed_%": round(100 * (1 - q["spots_out"] / q["spots_in"]), 1),
                 "low_quality_kept": q["n_low_quality_kept"], "high_mito_kept": q["n_high_mito_kept"],
                 "mito_cut_%": round(q["mito_cutoff_pct"], 1), "genes_out": q["genes_out"],
                 "NN_um": round(float(np.median(d[:, 1])), 1)})
    raw.obs["removed"] = (~raw.obs_names.isin(kept.obs_names)).astype(str)
    maps.append((s["id"], raw))

display(pd.DataFrame(rows))

fig, axes = plt.subplots(len(maps), 3, figsize=(14, 3.2 * len(maps)))
for (pid, raw), ax in zip(maps, axes):
    ax[0].hist(np.log1p(raw.obs["total_counts"]), 60); ax[0].axvline(np.log1p(cfg["st"]["min_counts"]), c="r"); ax[0].set_title(f"{pid} · log1p counts")
    ax[1].hist(np.log1p(raw.obs["n_genes_by_counts"]), 60); ax[1].axvline(np.log1p(cfg["st"]["min_genes"]), c="r"); ax[1].set_title("log1p genes")
    xy = raw.obsm["spatial"]; rm = raw.obs["removed"] == "True"
    ax[2].scatter(xy[~rm, 0], -xy[~rm, 1], s=2, c="lightgrey"); ax[2].scatter(xy[rm, 0], -xy[rm, 1], s=4, c="royalblue")
    ax[2].set_aspect("equal"); ax[2].set_axis_off(); ax[2].set_title(f"removed: {rm.sum()}")
plt.tight_layout()

## 8 · Save all patients

In [ ]:
SAVE = True
out = Path(cfg["paths"]["processed_st"])
if SAVE:
    out.mkdir(parents=True, exist_ok=True)
    for s in cfg["slices"]:
        a = pst.preprocess_st(pst.load_visium(str(outs_dir(s["st"])), s["id"]), cfg, slice_cfg=s)
        validate_adata(a, "st")
        a.write_h5ad(out / f"{s['id']}.h5ad")
        print("saved", s["id"], a.shape)

## Notes / open questions
- Patient IDs: the intern used `Patient_1..6`. SM pairing will use UKF ids; confirm the mapping once SM arrives.
- `H&E/*.jpg` is a separate full-resolution image (not used yet; candidate for histology-based references, P5).
- Extra CSVs (UKF248, UKF275): decide after §0b.
- 